# Experiment 20 — V1 / V2 LLM Annotation of 10 Struggling Students

Parametrized rerun of the four pre-V3 prompt configurations on the same 10-student
cohort that V3 (exp19) was run on, so all five prompts can be compared head-to-head
against the same two human raters (Pranay + Arundhati).

## Variants (set `PROMPT_VARIANT` in the config cell)

| Variant | Prompt builder | Mental model? |
|---|---|---|
| `v1_baseline` | `build_curriculum_aware_prompt` | No |
| `v1_enriched` | `build_curriculum_aware_prompt` + mental model | Yes |
| `v2_baseline` | `build_curriculum_aware_prompt_v2` | No |
| `v2_enriched` | `build_curriculum_aware_prompt_v2` + mental model | Yes |

## Paths
- Student code inputs: `scripts/annotation_tool/annotation_inputs/student_{sid}.json`
- Problem KC mapping: `dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv`
- Output (per student): `results/human_validation/llm_{variant}_10students/llm_{variant}_annotations_{sid}.json`

## Methodology notes
- Per-submission API calls (one problem per call, `focus_problem_ids=[pid]`).
- Perfect-score submissions (score == 1.0) are skipped client-side; recorded as empty gaps with no API call. Matches exp19/V3.
- KC tag extraction: only `student_analysis[].knowledge_gaps[].missing_concept` is taken, because that is what human raters annotate (per-problem current gaps). `future_predictions[].at_risk_topic` is *not* counted by default — it represents downstream risk, not current gap, and including it would inflate the LLM’s tag set unfairly vs. humans. Toggle `INCLUDE_FUTURE_PREDICTIONS` if you want the legacy exp10a/exp11 behaviour for sanity-comparison.
- Checkpoint per student: if the output file exists, that student is skipped on rerun. To force re-run, delete the file.

In [ ]:
# =====================================================================
# CONFIGURATION — change PROMPT_VARIANT and re-run-all to switch prompts.
# =====================================================================

PROMPT_VARIANT = "v2_enriched"  # one of: v1_baseline, v1_enriched, v2_baseline, v2_enriched

MODEL_ID = "gemini-2.5-flash"
TEMPERATURE = 0.3
SLEEP_SECONDS = 3.0  # paid tier safe pacing
INCLUDE_FUTURE_PREDICTIONS = False  # True = legacy exp10a/exp11 extraction

STUDENT_IDS = [10155, 9948, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499]

assert PROMPT_VARIANT in {"v1_baseline", "v1_enriched", "v2_baseline", "v2_enriched"}, (
    f"Unknown PROMPT_VARIANT: {PROMPT_VARIANT}"
)

VARIANT_TO_RATER = {
    "v1_baseline": "LLM_Gemini_V1_Baseline",
    "v1_enriched": "LLM_Gemini_V1_Enriched",
    "v2_baseline": "LLM_Gemini_V2_Baseline",
    "v2_enriched": "LLM_Gemini_V2_Enriched",
}
RATER_LABEL = VARIANT_TO_RATER[PROMPT_VARIANT]
USES_MENTAL_MODEL = PROMPT_VARIANT.endswith("_enriched")
USES_V2_PROMPT = PROMPT_VARIANT.startswith("v2_")

print(f"PROMPT_VARIANT = {PROMPT_VARIANT}")
print(f"  rater label = {RATER_LABEL}")
print(f"  uses V2 prompt = {USES_V2_PROMPT}")
print(f"  uses mental model = {USES_MENTAL_MODEL}")
print(f"  include future_predictions in gap extraction = {INCLUDE_FUTURE_PREDICTIONS}")
print(f"  sleep between calls = {SLEEP_SECONDS}s")

PROMPT_VARIANT = v2_baseline
  rater label = LLM_Gemini_V2_Baseline
  uses V2 prompt = True
  uses mental model = False
  include future_predictions in gap extraction = False
  sleep between calls = 3.0s


In [2]:
# Imports & paths
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types

ROOT = Path("/mnt/d/Projects/kintsugi")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.prompts import (
    build_curriculum_aware_prompt,
    build_curriculum_aware_prompt_v2,
)
from lib.llm_batch_analyzer import format_submissions, clean_json_response
from lib.mental_model import (
    load_skill_map,
    build_prerequisite_graph,
    calculate_student_profile,
    get_weak_skills,
    build_mental_model_payload,
)
from lib.experiment_utils import load_best_attempts_df
from utils.dataset import load_topics_json, load_problem_descriptions
from utils.constants import GEMINI_API_KEY

API_KEY = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY")
if not API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running.")

INPUT_DIR = ROOT / "scripts" / "annotation_tool" / "annotation_inputs"
OUTPUT_DIR = ROOT / "results" / "human_validation" / f"llm_{PROMPT_VARIANT}_10students"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KC_COLUMNS = [
    "If/Else", "NestedIf", "While", "For", "NestedFor",
    "Math+-*/", "Math%", "LogicAndNotOr", "LogicCompareNum", "LogicBoolean",
    "StringFormat", "StringConcat", "StringIndex", "StringLen",
    "StringEqual", "CharEqual", "ArrayIndex", "DefFunction",
]
VALID_KCS = set(KC_COLUMNS)

print(f"Output dir: {OUTPUT_DIR}")

Output dir: /mnt/d/Projects/kintsugi/results/human_validation/llm_v2_baseline_10students


In [3]:
# Curriculum context (topics + problem descriptions) shared across all variants.
topics = load_topics_json() or {}
problem_descriptions = load_problem_descriptions() or {}

# Mental models (only needed for enriched variants, but cheap to build either way).
mental_models = {}
if USES_MENTAL_MODEL:
    print("Building mental models for enriched variant...")
    attempts_df = load_best_attempts_df()
    skill_map, all_skills = load_skill_map("dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv")
    prereq_graph = build_prerequisite_graph()
    for sid in STUDENT_IDS:
        profile = calculate_student_profile(sid, attempts_df, skill_map, all_skills)
        weak_pairs = get_weak_skills(profile)
        mental_models[sid] = build_mental_model_payload(sid, profile, weak_pairs, prereq_graph)
        print(f"  {sid}: {len(weak_pairs)} weak skills → {[s for s, _ in weak_pairs[:5]]}{'...' if len(weak_pairs) > 5 else ''}")
else:
    print("Skipping mental model construction (baseline variant).")

Skipping mental model construction (baseline variant).


In [4]:
# Per-problem prompt builder (dispatches on variant).
def build_prompt(student_id: int, problem_id: int) -> str:
    if USES_V2_PROMPT:
        base = build_curriculum_aware_prompt_v2(topics, problem_descriptions, [problem_id])
    else:
        base = build_curriculum_aware_prompt(topics, problem_descriptions, [problem_id])
    if not USES_MENTAL_MODEL:
        return base
    ctx = json.dumps(mental_models[student_id], indent=2)
    return (
        base
        + "\n\nAdditional Student Mental Model Context (high-priority):\n"
        + ctx
        + "\n\nUse this context to better judge likely misconceptions and future risks."
    )

# Quick sanity check on prompt size for the first student/problem in the cohort.
_sample_prompt = build_prompt(STUDENT_IDS[0], 1)
print(f"Sample prompt length: {len(_sample_prompt):,} chars (~{len(_sample_prompt)//4:,} tokens est.)")
print(f"First 600 chars:\n{_sample_prompt[:600]}")

Sample prompt length: 13,647 chars (~3,411 tokens est.)
First 600 chars:
You are an expert CS1 instructor analyzing individual student code submissions to identify knowledge gaps and predict future struggles.

COURSE CURRICULUM:
{
  "course_info": {
    "course_name": "CS 1114 - Introduction to Java Programming",
    "level": "CS1",
    "language": "Java",
    "textbook": "Introduction to Java Programming",
    "typical_coverage": "Chapters 1-9"
  },
  "topics": [
    {
      "chapter": 1,
      "chapter_name": "Introduction to Computers, Programs, and Java",
      "topics": [
        "What is a computer and how it works",
        "Programs and programming language


In [5]:
# Gemini client.
client = genai.Client(api_key=API_KEY)

def call_gemini(system_instruction: str, user_content: str) -> str:
    """V1/V2 invocation pattern: prompt as system_instruction, submission as user content."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=user_content,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=TEMPERATURE,
            response_mime_type="application/json",
        ),
    )
    return response.text or ""

print("Gemini client configured.")

Gemini client configured.


In [6]:
# Parsing & KC extraction (handles both nested student_analysis schema and any flat fallback).
def parse_llm_response(raw_text: str):
    cleaned = clean_json_response(raw_text or "")
    try:
        return json.loads(cleaned), "ok"
    except json.JSONDecodeError as e:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start != -1 and end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1]), "ok_extracted_json"
            except json.JSONDecodeError as e2:
                return {"raw": raw_text}, f"parse_error: {e2}"
        return {"raw": raw_text}, f"parse_error: {e}"

def extract_kc_tags(parsed):
    """Return (valid_kcs_sorted, invalid_kcs_sorted). Only `missing_concept` by default;
    include `at_risk_topic` if INCLUDE_FUTURE_PREDICTIONS is True."""
    if not isinstance(parsed, dict):
        return [], []
    valid, invalid = set(), set()

    def _consume_tag(tag):
        t = str(tag or "").strip()
        if not t:
            return
        (valid if t in VALID_KCS else invalid).add(t)

    analyses = parsed.get("student_analysis", [])
    if isinstance(analyses, dict):
        analyses = [analyses]
    elif not isinstance(analyses, list):
        analyses = []
    for a in analyses:
        if not isinstance(a, dict):
            continue
        for gap in a.get("knowledge_gaps", []) or []:
            _consume_tag(gap.get("missing_concept") if isinstance(gap, dict) else gap)
        if INCLUDE_FUTURE_PREDICTIONS:
            for pred in a.get("future_predictions", []) or []:
                _consume_tag(pred.get("at_risk_topic") if isinstance(pred, dict) else pred)

    # Flat fallback in case the model emitted a top-level shape.
    for gap in parsed.get("knowledge_gaps", []) or []:
        _consume_tag(gap.get("missing_concept") if isinstance(gap, dict) else gap)
    if INCLUDE_FUTURE_PREDICTIONS:
        for pred in parsed.get("future_predictions", []) or []:
            _consume_tag(pred.get("at_risk_topic") if isinstance(pred, dict) else pred)

    return sorted(valid), sorted(invalid)

In [7]:
# Load student inputs and report planned API-call count.
student_data = {}
for sid in STUDENT_IDS:
    with open(INPUT_DIR / f"student_{sid}.json", "r") as f:
        student_data[sid] = json.load(f)
    subs = student_data[sid]["submissions"]
    n_total = len(subs)
    n_perfect = sum(1 for v in subs.values() if v["score"] >= 1.0)
    print(f"Student {sid}: {n_total} problems ({n_perfect} perfect, {n_total - n_perfect} to annotate)")

total_calls = sum(
    sum(1 for v in d["submissions"].values() if v["score"] < 1.0)
    for d in student_data.values()
)
print(f"\nTotal API calls needed: {total_calls}")
print(f"Estimated runtime at {SLEEP_SECONDS}s sleep: {(total_calls * SLEEP_SECONDS) / 60:.0f} min")

Student 10155: 46 problems (11 perfect, 35 to annotate)
Student 9948: 39 problems (24 perfect, 15 to annotate)
Student 14189: 48 problems (18 perfect, 30 to annotate)
Student 14352: 41 problems (25 perfect, 16 to annotate)
Student 14362: 26 problems (19 perfect, 7 to annotate)
Student 14363: 28 problems (17 perfect, 11 to annotate)
Student 14374: 33 problems (10 perfect, 23 to annotate)
Student 14414: 41 problems (26 perfect, 15 to annotate)
Student 14474: 32 problems (15 perfect, 17 to annotate)
Student 14499: 38 problems (19 perfect, 19 to annotate)

Total API calls needed: 188
Estimated runtime at 3.0s sleep: 9 min


In [8]:
# Main run loop — per student, save-after-each, skip if output already exists.
all_results = {}

for student_idx, sid in enumerate(STUDENT_IDS):
    output_path = OUTPUT_DIR / f"llm_{PROMPT_VARIANT}_annotations_{sid}.json"
    if output_path.exists():
        print(f"\n[{student_idx+1}/{len(STUDENT_IDS)}] Student {sid} — ALREADY DONE, skipping.")
        with open(output_path, "r") as f:
            all_results[sid] = json.load(f)
        continue

    print(f"\n{'='*60}\n[{student_idx+1}/{len(STUDENT_IDS)}] Student {sid}\n{'='*60}")
    submissions = student_data[sid]["submissions"]
    annotations, raw_responses, errors = {}, {}, []
    call_count = 0

    for pid_str in sorted(submissions.keys(), key=lambda x: int(x)):
        pid = int(pid_str)
        sub = submissions[pid_str]
        code, score = sub["code"], sub["score"]

        if score >= 1.0:
            annotations[pid_str] = {"gaps": []}
            continue

        system_instruction = build_prompt(sid, pid)
        user_content = format_submissions([{
            "Code": code,
            "SubjectID": sid,
            "ProblemID": pid,
            "Score": score,
            "Attempt": sub.get("attempt", 1),
            "Compile.Result": sub.get("compile_result", "Unknown"),
        }])

        call_count += 1
        try:
            t0 = time.time()
            raw = call_gemini(system_instruction, user_content)
            elapsed = time.time() - t0
            parsed, status = parse_llm_response(raw)
            valid_kcs, invalid_kcs = extract_kc_tags(parsed)
            annotations[pid_str] = {"gaps": valid_kcs}
            raw_responses[pid_str] = {
                "raw_response": raw,
                "parsed_response": parsed,
                "parse_status": status,
                "time_sec": round(elapsed, 3),
                "score": score,
                "invalid_kcs": invalid_kcs,
            }
            gap_str = ", ".join(valid_kcs) if valid_kcs else "(none)"
            print(f"  P{pid} (score={score:.2f}) → [{gap_str}] ({elapsed:.1f}s)")
        except Exception as e:
            err = f"API error on P{pid}: {e}"
            print(f"  ERROR: {err}")
            errors.append(err)
            annotations[pid_str] = {"gaps": []}
            raw_responses[pid_str] = {"error": str(e), "score": score}
        time.sleep(SLEEP_SECONDS)

    result = {
        "rater": RATER_LABEL,
        "studentId": str(sid),
        "student_id": str(sid),
        "prompt_variant": PROMPT_VARIANT,
        "model_id": MODEL_ID,
        "temperature": TEMPERATURE,
        "include_future_predictions": INCLUDE_FUTURE_PREDICTIONS,
        "exportDate": datetime.now().isoformat(),
        "total_problems": len(submissions),
        "totalAnnotated": len(annotations),
        "total_calls": call_count,
        "errors": errors,
        "annotations": annotations,
        "raw_responses": raw_responses,
    }
    with open(output_path, "w") as f:
        json.dump(result, f, indent=2)
    all_results[sid] = result

    n_with_gaps = sum(1 for a in annotations.values() if a.get("gaps"))
    print(f"\n  SAVED: {output_path.name}")
    print(f"  Calls: {call_count} | With gaps: {n_with_gaps} | Errors: {len(errors)}")

print(f"\n{'='*60}\nALL STUDENTS COMPLETE — {PROMPT_VARIANT}\n{'='*60}")


[1/10] Student 10155
  P3 (score=0.81) → [If/Else, LogicAndNotOr] (23.7s)
  P22 (score=0.36) → [DefFunction, If/Else, LogicAndNotOr, LogicCompareNum] (13.4s)
  P24 (score=0.59) → [If/Else, LogicAndNotOr, LogicCompareNum] (21.2s)
  P28 (score=0.20) → [If/Else, StringConcat, StringIndex] (9.8s)
  P31 (score=0.27) → [For, StringConcat, StringIndex] (10.0s)
  P32 (score=0.00) → [DefFunction] (17.8s)
  P33 (score=0.13) → [(none)] (15.5s)
  P34 (score=0.50) → [(none)] (7.5s)
  P36 (score=0.41) → [(none)] (6.4s)
  P37 (score=0.67) → [LogicAndNotOr, StringEqual, StringIndex] (16.9s)
  P38 (score=0.73) → [For, LogicAndNotOr, StringIndex] (23.2s)
  P39 (score=0.63) → [(none)] (3.0s)
  P40 (score=0.15) → [If/Else, StringIndex] (11.0s)
  P43 (score=0.62) → [ArrayIndex] (25.6s)
  P44 (score=0.40) → [ArrayIndex, For, If/Else, Math%] (11.5s)
  P45 (score=0.13) → [(none)] (2.2s)
  P46 (score=0.56) → [ArrayIndex, For, If/Else, LogicAndNotOr] (20.0s)
  P48 (score=0.27) → [ArrayIndex, For] (10.8s)
  P49

In [9]:
# Summary table.
print(f"\n{'='*60}\nSUMMARY — {PROMPT_VARIANT}\n{'='*60}")
print(f"\n{'StudentID':>10} {'Total':>6} {'Calls':>6} {'WithGaps':>9} {'Errors':>7}")
print("-" * 45)
total_calls_all = total_gaps_all = 0
for sid in STUDENT_IDS:
    if sid not in all_results:
        print(f"{sid:>10}  — NOT RUN")
        continue
    r = all_results[sid]
    ann = r["annotations"]
    n_total = r.get("total_problems", len(ann))
    n_calls = r.get("total_calls", len(r.get("raw_responses", {})))
    n_with_gaps = sum(1 for a in ann.values() if a.get("gaps"))
    n_errors = len(r.get("errors", []))
    total_calls_all += n_calls
    total_gaps_all += n_with_gaps
    print(f"{sid:>10} {n_total:>6} {n_calls:>6} {n_with_gaps:>9} {n_errors:>7}")
print("-" * 45)
print(f"{'TOTAL':>10} {'':>6} {total_calls_all:>6} {total_gaps_all:>9}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")


SUMMARY — v2_baseline

 StudentID  Total  Calls  WithGaps  Errors
---------------------------------------------
     10155     46     35        21       0
      9948     39     15         9       0
     14189     48     30        18       0
     14352     41     16        15       0
     14362     26      7         7       0
     14363     28     11        11       0
     14374     33     23        23       0
     14414     41     15        15       0
     14474     32     17        15       0
     14499     38     19        12       0
---------------------------------------------
     TOTAL           188       146

All outputs saved to: /mnt/d/Projects/kintsugi/results/human_validation/llm_v2_baseline_10students


In [10]:
# Quick sanity check against human annotations for student 10155.
HUMAN_DIR = ROOT / "dataset" / "Rater_KC_Tags" / "Rated_KC_V3"
pranay_files = list(HUMAN_DIR.glob("kc_annotations_Pranay Ghuge_10155_*.json"))
if pranay_files and 10155 in all_results:
    with open(pranay_files[0], "r") as f:
        human_ann = json.load(f)
    llm_ann = all_results[10155].get("annotations", {})
    print(f"Sanity check — Student 10155, first 5 non-empty problems ({PROMPT_VARIANT} vs human):")
    print(f"\n{'PID':>5} {'Human':<45} {'LLM':<45}")
    print("-" * 100)
    count = 0
    for pid_str in sorted(human_ann.get("annotations", {}).keys(), key=lambda x: int(x)):
        h_gaps = human_ann["annotations"][pid_str].get("gaps", [])
        l_gaps = llm_ann.get(pid_str, {}).get("gaps", [])
        if not h_gaps and not l_gaps:
            continue
        print(f"{pid_str:>5} {(', '.join(h_gaps) or '(none)'):<45} {(', '.join(l_gaps) or '(none)'):<45}")
        count += 1
        if count >= 5:
            break
else:
    print("No human annotation found for 10155 or student not run — skipping sanity check.")
print("\nDone. Repeat with PROMPT_VARIANT set to the other three values to complete the matrix.")

Sanity check — Student 10155, first 5 non-empty problems (v2_baseline vs human):

  PID Human                                         LLM                                          
----------------------------------------------------------------------------------------------------
    3 LogicCompareNum, LogicAndNotOr, LogicBoolean  If/Else, LogicAndNotOr                       
   22 DefFunction, LogicCompareNum, LogicAndNotOr   DefFunction, If/Else, LogicAndNotOr, LogicCompareNum
   24 LogicCompareNum, LogicAndNotOr, Math+-*/      If/Else, LogicAndNotOr, LogicCompareNum      
   28 LogicAndNotOr, StringFormat, StringIndex, StringConcat If/Else, StringConcat, StringIndex           
   31 (none)                                        For, StringConcat, StringIndex               

Done. Repeat with PROMPT_VARIANT set to the other three values to complete the matrix.
